Here we will be implementing BCH, APC codes for Mersenne Primes

- We will be usimg a lookup table for Syndromes to find where the error has occured

- Construction of BCH codes to achieve the cyclic codes

- Construction of ACPC codes to achieve cyclically permutable codes. 

In [127]:
import numpy as np
import math
import galois
import random

In [2]:
# mersenne number n = 2**m-1
# mersenne prime: mersenne number should be prime

m = 5
n = 2**m - 1
print(f'Block length: {n}')

Block length: 31


In [11]:
# dir(galois.GF2)

BCH codes using built-in functions

In [75]:
bch = galois.BCH(n, 21)

msg = [1, 0, 1]*7

codeword = bch.encode(msg)
print(f'BCH encoded Codeword: {codeword}')

decoded = bch.decode(codeword)
print(f'\n Decode message from codeword:{decoded}')

BCH encoded Codeword: [1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 1 1 1 0 0 1 1 1 0]

 Decode message from codeword:[1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1]


In [ ]:
r = codeword.copy()

print(r)
r[3] ^= 1
r[17] ^= 1
print(r)
decoded = bch.decode(r)
print(f'\n Decode message from codeword:{decoded}')

[1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 1 1 1 0 0 1 1 1 0]
[1 0 1 0 0 1 1 0 1 1 0 1 1 0 1 1 0 0 1 0 1 1 1 1 1 0 0 1 1 1 0]

 Decode message from codeword:[1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1]


In [24]:
print(bch.generator_poly)
print(bch.alpha)
print(bch.roots)
print(bch.t)

x^10 + x^9 + x^8 + x^6 + x^5 + x^3 + 1
2
[ 2  4  8 16]
2


Encoding Manually

In [25]:
# construction of GF(32) over GF(2)
GF = galois.GF(2**m)

irr_poly = GF.irreducible_poly
alpha = GF.primitive_element

print(f'Irreducble Polynomial: {irr_poly}\n')
print(f"Primitive Element: {alpha}")

Irreducble Polynomial: x^5 + x^2 + 1

Primitive Element: 2


In [27]:
#circular shift
c = np.array([1,0,1,1,0,0])
print(np.roll(c,1))
print(np.roll(c,4))

[0 1 0 1 1 0]
[1 1 0 0 1 0]


In [35]:
l = GF.properties
print(l)

Galois Field:
  name: GF(2^5)
  characteristic: 2
  degree: 5
  order: 32
  irreducible_poly: x^5 + x^2 + 1
  is_primitive_poly: True
  primitive_element: x


Phase 1: Construct BCH(31,21,5)

In [36]:
bch = galois.BCH(31,21)
print(bch)

BCH Code:
  [n, k, d]: [31, 21, 5]
  field: GF(2)
  extension_field: GF(2^5)
  generator_poly: x^10 + x^9 + x^8 + x^6 + x^5 + x^3 + 1
  is_primitive: True
  is_narrow_sense: True
  is_systematic: True


In [39]:
GF = bch.field
print(GF)
alpha = GF.primitive_element
print(f'Primitive Element: {alpha}')

<class 'galois.GF(2, primitive_element='1', irreducible_poly='x + 1')'>
Primitive Element: 1


In [42]:
GF_32 = bch.extension_field
print(GF_32)
alpha_32 = GF_32.primitive_element
print(f'Primitive Element: {alpha_32}')

<class 'galois.GF(2^5, primitive_element='x', irreducible_poly='x^5 + x^2 + 1')'>
Primitive Element: 2


In [59]:
gen_poly = bch.generator_poly
print(gen_poly)
print(f'Degree of Generator Polynomial: {gen_poly.degree}')
roots = bch.roots
print(f'Roots of BCH: {roots}')

print(f'Field of Generator Polynomial: {gen_poly.field}')

print(f'Type of roots: {type(roots[0])}')
print(roots[0])
print(bch.field)

x^10 + x^9 + x^8 + x^6 + x^5 + x^3 + 1
Degree of Generator Polynomial: 10
Roots of BCH: [ 2  4  8 16]
Field of Generator Polynomial: <class 'galois.GF(2, primitive_element='1', irreducible_poly='x + 1')'>
Type of roots: <class 'galois.GF(2^5, primitive_element='x', irreducible_poly='x^5 + x^2 + 1')'>
2
<class 'galois.GF(2, primitive_element='1', irreducible_poly='x + 1')'>


In [62]:
for i in bch.roots:
    print(gen_poly(i, field=GF_32))

0
0
0
0


Computing hout(x)

In [73]:
GF2 = galois.GF(2)

x = galois.Poly.Identity(GF2)

x_n = x**n + GF2(1)

h_out = x_n // gen_poly

print(f'Parity-check polynomial: {h_out}')
print(f'Degree of polynomial: {h_out.degree}')

print(h_out * gen_poly == x_n)

Parity-check polynomial: x^21 + x^20 + x^18 + x^16 + x^14 + x^13 + x^12 + x^11 + x^8 + x^5 + x^3 + 1
Degree of polynomial: 21
True


In [98]:
Hout = bch.H
print(Hout.shape)

(10, 31)


In [121]:
factors, _ = galois.factors(x_n)
# for f in factors:
#     print(f)
print(factors)
gen_fac,_ = galois.factors(gen_poly)
print(gen_fac)

choice_gin = [ i for i in factors[1:] if i not in gen_fac ]
print(choice_gin)

[Poly(x + 1, GF(2)), Poly(x^5 + x^2 + 1, GF(2)), Poly(x^5 + x^3 + 1, GF(2)), Poly(x^5 + x^3 + x^2 + x + 1, GF(2)), Poly(x^5 + x^4 + x^2 + x + 1, GF(2)), Poly(x^5 + x^4 + x^3 + x + 1, GF(2)), Poly(x^5 + x^4 + x^3 + x^2 + 1, GF(2))]
[Poly(x^5 + x^2 + 1, GF(2)), Poly(x^5 + x^4 + x^3 + x^2 + 1, GF(2))]
[Poly(x^5 + x^3 + 1, GF(2)), Poly(x^5 + x^3 + x^2 + x + 1, GF(2)), Poly(x^5 + x^4 + x^2 + x + 1, GF(2)), Poly(x^5 + x^4 + x^3 + x + 1, GF(2))]


In [119]:
x = np.arange(10)
y = np.arange(4,10)
x_y = [i for i in x if i not in y]
print(x_y)

[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [128]:
gin = random.choice(choice_gin)
print(f"Inner code polynomial chosen: {gin}")

Inner code polynomial chosen: x^5 + x^3 + x^2 + x + 1


ACPC Encoder

In [137]:
msg = galois.GF2.Random(16)
print(msg)
print(type(msg))
msg1 = [ np.random.randint(2) for i in range(16) ]
print(msg1)

[0 0 0 1 1 1 0 1 0 0 1 0 0 1 0 0]
<class 'galois.GF(2, primitive_element='1', irreducible_poly='x + 1')'>
[0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0]


In [ ]:
M = galois.Poly(msg1[::-1])
print(M)
I = M * gin + GF2(1)
C = I * gen_poly
C %= x_n
print(C.coeffs)

x^14 + x^12 + x^11 + x^8 + x^7 + x^6 + x^4 + x^3 + x^2
30


In [149]:
c = C.coeffs

r = np.roll(c, 3)
print(r)

c_hat = bch.decode(r)
print(c_hat)

[1 0 1 1 1 1 0 1 0 0 0 1 1 0 0 1 1 1 1 1 0 0 0 1 1 0 0 1 1 1]
[1 0 1 1 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 1]


In [147]:
for l in range(31):

    v_hat = np.roll(c_hat, -l)

    syndrome = Hout @ v_hat % 2

    if np.all(syndrome == 0):
        correct_shift = l
        v = v_hat
        break

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 20 is different from 31)